# 🤖 Robotics Warehouse System — Hierarchical Agent Architecture

## From Simple LLM Calls to Multi-Agent Delegation 🏭

This notebook takes you on a progressive journey through agent complexity — starting with a bare LLM call, building up to a simple tool-equipped agent, and culminating in a **hierarchical multi-agent warehouse system** where agents delegate tasks down the chain.

### 🎯 What You'll Learn

1. **🧠 Raw LLM Invocation** - Calling an LLM directly without tools or agents
2. **🔧 Simple Agent with Tools** - Building a single agent with a custom tool (weather check)
3. **🏗️ Hierarchical Agent System** - Creating a 3-tier warehouse management hierarchy:
   - **Warehouse Manager** → delegates to Zone Manager
   - **Zone Manager** → delegates to Robot Controller
   - **Robot Controller** → executes the actual task
4. **📊 Streaming & Message Inspection** - Observing the full delegation chain in real time

### 🌟 Key Architecture Pattern: Nested Agent Delegation

```
   User Request: "Move packages from Zone A to Zone B"
              │
    ┌─────────▼──────────┐
    │ Warehouse Manager  │  ← Top-level: receives user tasks
    │  (uses tool that   │
    │   calls Zone Mgr)  │
    └─────────┬──────────┘
              │ tool call
    ┌─────────▼──────────┐
    │   Zone Manager     │  ← Mid-level: delegates to robots
    │  (uses tool that   │
    │   calls Robot Ctrl)│
    └─────────┬──────────┘
              │ tool call
    ┌─────────▼──────────┐
    │  Robot Controller   │  ← Execution level: runs the task
    └────────────────────┘
```

### 📋 Prerequisites
- **OpenAI API key** for GPT-4o access
- Basic understanding of LangChain agents and tools
- Familiarity with function calling concepts

---

*💡 **Pro Tip**: This notebook uses `create_agent()` from LangChain, which creates a ReAct-style agent. Each agent can call tools — and those tools can invoke other agents, creating the hierarchical chain!*

In [ ]:
# 📦 Install/upgrade required packages
%pip install --quiet -U langchain_core langgraph langchain_openai

## 1. 🔧 Environment Setup

### 📦 Required Packages
- **`langchain_core`** - Foundation library with message types and tool abstractions
- **`langgraph`** - Core framework for stateful agent workflows
- **`langchain_openai`** - OpenAI GPT model integration for agent reasoning

## 2. 🧠 Part 1: Raw LLM Invocation (No Agent, No Tools)

### 📋 Starting Simple — Direct LLM Calls

Before we build agents, let's see how to call an LLM directly. This is the simplest form of interaction — you send a message, the model responds. No tools, no state, no workflows.

### 💡 Why Start Here?
Understanding the raw LLM call helps you appreciate what agents *add*:
- **Tools** → let the LLM take actions (search, calculate, book)
- **State** → let the LLM remember previous interactions
- **Workflows** → let multiple LLMs coordinate complex tasks

In [ ]:
import os, getpass

def _set_env(var: str):
    """Helper to prompt for an environment variable if not already set."""
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

# 🔑 Prompt for the OpenAI API key (only if not already in environment)
_set_env("OPENAI_API_KEY")
print("✅ API key configured!")

In [ ]:
# ============================================================
# 🤖 Initialize the LLM model and import create_agent
# ============================================================
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent  # Creates ReAct-style agents with tools

# GPT-4o: Fast, capable model for reasoning and tool calling
model = ChatOpenAI(model="gpt-4o")

print("✅ Model initialized!")

In [ ]:
# ============================================================
# 🧠 Direct LLM call — no agent, no tools, just a chat
# ============================================================
# model.invoke() sends a message directly to GPT-4o and returns
# the raw response. This is the simplest possible LLM interaction.
# Notice: no tools available, no memory, no state tracking.
# ============================================================
model.invoke([{"role": "user", "content": "Good morning!"}])

## 3. 🔧 Part 2: Simple Agent with a Tool

### 🚀 Upgrading from Raw LLM to an Agent

Now let's level up! We'll create a **simple agent** that can use a **tool** (weather check). This demonstrates the core pattern:

1. **Define a tool** — a Python function the LLM can call
2. **Create an agent** — wrap the LLM with tools and a system prompt
3. **Run the agent** — the agent reasons about which tool to use, calls it, and returns the result

### 🔑 Key Concept: ReAct Pattern
`create_agent()` creates a **ReAct** (Reason + Act) agent. The LLM:
1. **Reasons** about what to do based on the user's message
2. **Acts** by calling a tool
3. **Observes** the tool's output
4. **Responds** with a final answer

This is the same pattern used by the warehouse agents later — just with simpler tools!

In [ ]:
# ============================================================
# 🌤️ Define a simple weather tool
# ============================================================
# This is a mock tool — in production, you'd call a real weather API.
# LangChain automatically converts the function signature + docstring
# into a tool schema for the LLM's function calling.
# ============================================================
def check_weather(location: str) -> str:
    '''Return the weather forecast for the specified location.'''
    return f"It's always sunny in {location}"

# ============================================================
# 🤖 Create a simple agent with the weather tool
# ============================================================
# create_agent() builds a ReAct-style agent that can:
#   1. Receive a user message
#   2. Decide whether to call a tool
#   3. Call the tool and observe the result
#   4. Return a final answer to the user
# ============================================================
weather_agent = create_agent(
    model=model,                          # The GPT-4o LLM
    tools=[check_weather],                # Tools available to this agent
    system_prompt="You are a helpful assistant",  # Agent's personality/role
)

print("✅ Weather agent created!")

### 🎨 Visualize the Weather Agent

Even a simple single-tool agent has a graph structure — the agent node connects to a tool node and back, forming the ReAct loop.

In [ ]:
# 🎨 Render the agent's graph — shows the ReAct loop structure
from IPython.display import Image, display
display(Image(weather_agent.get_graph().draw_mermaid_png()))

### 🧪 Test the Weather Agent

Let's test the agent with a weather question. Watch the stream output to see the ReAct loop in action:
1. Agent receives message → 2. Calls `check_weather` tool → 3. Gets result → 4. Responds to user

In [ ]:
# ============================================================
# 🧪 Test the weather agent with streaming output
# ============================================================
# stream() with stream_mode="updates" shows each step as it happens:
#   - The model's reasoning and tool call decision
#   - The tool execution result
#   - The model's final response
# ============================================================
inputs = {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
for chunk in weather_agent.stream(inputs, stream_mode="updates"):
    print(chunk)

In [ ]:
# ============================================================
# 📋 Alternative: invoke() returns the complete final result
# ============================================================
# Unlike stream(), invoke() waits for the entire agent execution
# to complete and returns all messages at once.
# ============================================================
response = weather_agent.invoke(inputs)

In [ ]:
# 📋 Pretty-print all messages in the agent's response
# Shows: user message → tool call → tool result → AI response
for m in response['messages']:
    m.pretty_print()

## 4. 🏭 Part 3: Hierarchical Warehouse Management System

### 🏗️ Building a 3-Tier Agent Hierarchy

Now we build the main event — a **hierarchical agent system** where three agents are nested inside each other:

```
Warehouse Manager → (tool calls) → Zone Manager → (tool calls) → Robot Controller
```

### 🔑 The Key Insight: Agents-as-Tools

The magic trick is that **each higher-level agent has a tool that invokes the lower-level agent**:
- The Warehouse Manager's tool calls `zone_manager.invoke()`
- The Zone Manager's tool calls `robot_controller.invoke()`
- The Robot Controller's tool just prints and executes the task

This creates a **nested delegation chain** — the user talks to the Warehouse Manager, who delegates down the hierarchy until the task is actually executed.

### 💡 Why This Pattern?
- **Separation of concerns** — each agent focuses on its level of responsibility
- **Scalability** — add more zone managers or robot controllers without changing the top level
- **Reusability** — each agent can be tested and used independently

### ⚠️ Building Bottom-Up
We must create agents from the **bottom up** (Robot Controller first, then Zone Manager, then Warehouse Manager) because each agent's tool needs to reference the agent below it.

### 🤖 Layer 1 (Bottom): Robot Controller

The Robot Controller is the **execution layer** — it receives a task description and performs it. This is the simplest agent in the hierarchy; its tool just confirms task execution.

In [ ]:
# ============================================================
# 🤖 Layer 1: Robot Controller — the execution level
# ============================================================
# This is the bottom of the hierarchy. When the Robot Controller
# receives a task, it "executes" it (in production, this would
# interface with actual robot hardware or a simulation API).
# ============================================================
from langchain_core.messages import HumanMessage

def robot_controller_tool(task: str)-> str:
    """Executes the given task."""
    # In production, this would send commands to a real robot
    print(f'🤖 Robot Controller executing task: {task}')
    return f"Executing task: {task}"

# Create the Robot Controller agent with its execution tool
robot_controller = create_agent(
    model=model,
    tools=[robot_controller_tool],
    system_prompt="You are the Robot Controller. Execute assigned tasks precisely.",
    name="robot_controller",
)

print("✅ Robot Controller agent created!")

### 👷 Layer 2 (Middle): Zone Manager

The Zone Manager receives tasks from the Warehouse Manager and **delegates them to the Robot Controller**. Its tool function actually calls `robot_controller.invoke()` — this is the key nesting pattern!

### 🔑 Agent-as-Tool Pattern
Notice that `zone_manager_tool` calls `robot_controller.invoke()` inside itself. This means when the Zone Manager LLM calls this tool, it triggers a **complete ReAct cycle** of the Robot Controller agent.

In [ ]:
# ============================================================
# 👷 Layer 2: Zone Manager — delegates tasks to Robot Controller
# ============================================================
# The Zone Manager's tool is special: it doesn't execute the
# task itself — it INVOKES the Robot Controller agent!
# This creates the nested delegation chain.
# ============================================================
def zone_manager_tool(task: str) -> str:
    """Delegates task to Robot Controller agent."""
    print(f'👷 Zone Manager delegating task: {task}')
    # 🔑 KEY PATTERN: Invoke another agent as a tool!
    # This triggers the Robot Controller's full ReAct cycle
    # HumanMessage wraps the task as input for the sub-agent
    return robot_controller.invoke({"messages": [HumanMessage(content=task)]})

# Create the Zone Manager agent with its delegation tool
zone_manager = create_agent(
    model=model,
    tools=[zone_manager_tool],
    system_prompt="You are the Zone Manager. Delegate given tasks to Robot Controllers.",
    name="zone_manager",
)

print("✅ Zone Manager agent created!")

### 🏢 Layer 3 (Top): Warehouse Manager

The Warehouse Manager is the **user-facing agent** at the top of the hierarchy. It receives tasks from the user and delegates them down through the Zone Manager, which in turn delegates to the Robot Controller.

### 🔗 The Complete Chain
```
User → Warehouse Manager → (tool) → Zone Manager → (tool) → Robot Controller → (tool) → Execute!
```

In [ ]:
# ============================================================
# 🏢 Layer 3: Warehouse Manager — top of the hierarchy
# ============================================================
# Same agent-as-tool pattern: the Warehouse Manager's tool
# invokes the Zone Manager agent, which in turn invokes
# the Robot Controller agent. Three levels of delegation!
# ============================================================
def warehouse_manager_tool(task: str) -> str:
    """Delegates task to Zone Manager agent."""
    print(f'🏢 Warehouse Manager delegating task: {task}')
    # Invoke the Zone Manager — starts the middle tier's ReAct cycle
    # The Zone Manager will then invoke the Robot Controller
    return zone_manager.invoke({"messages": [HumanMessage(content=task)]})

# Create the Warehouse Manager agent — the entry point for user tasks
warehouse_manager = create_agent(
    model=model,
    tools=[warehouse_manager_tool],
    system_prompt="You are the Warehouse Manager. Given the task by the user, Delegate tasks to Zone Managers.",
    name="warehouse_manager",
)

print("✅ Warehouse Manager agent created — full hierarchy ready!")

## 5. 🧪 Run the Hierarchical Warehouse System

### 🚀 Testing the Full Delegation Chain

Let's give the Warehouse Manager a real task and watch it delegate down through all three layers!

In [ ]:
# ============================================================
# 📋 Define the task for the warehouse system
# ============================================================
# This task will flow through: Warehouse Manager → Zone Manager → Robot Controller
task = "Move all packages from Zone A to Zone B."
print(f"📋 Task to execute: '{task}'")

### 🎨 Visualize the Warehouse Manager's Graph

Note: This only shows the Warehouse Manager's local graph (model → tools loop). The nested agents inside the tools are not visible in this view — they run as sub-processes within the tool calls.

In [ ]:
# 🎨 Render the Warehouse Manager's graph structure
from IPython.display import Image, display
display(Image(warehouse_manager.get_graph().draw_mermaid_png()))

### 🚀 Execute the Task — Watch the Delegation Chain

When you run the next cell, watch the print statements to see the full delegation chain:
1. 🏢 Warehouse Manager receives the task
2. 👷 Zone Manager gets delegated the task
3. 🤖 Robot Controller executes the task

In [ ]:
# ============================================================
# 📝 Alternative streaming approach (commented out)
# ============================================================
# This shows how to use stream_mode="updates" instead of "messages"
# Uncomment to try it — "updates" shows node-level updates rather
# than individual messages
# ============================================================
# response = warehouse_manager.stream({"role": "user", "content": f"Delegate task: {task}"}, stream_mode="updates") 
# for chunk in response:
#     print(chunk)

In [ ]:
# ============================================================
# 🚀 Run the full hierarchical delegation
# ============================================================
# invoke() triggers the Warehouse Manager, which calls its tool,
# which invokes the Zone Manager, which calls its tool,
# which invokes the Robot Controller — a 3-deep chain!
# ============================================================
inputs = {"messages": [{"role": "user", "content": f"Delegate task: {task}"}]}
response = warehouse_manager.invoke(inputs)

# Pretty-print all messages to see the Warehouse Manager's perspective
print("\n📋 Warehouse Manager's message history:")
for m in response['messages']:
    m.pretty_print()

### 📊 Detailed Streaming — Inspect Every Step

The cell below uses `stream(stream_mode="messages")` for fine-grained visibility into what happens inside the Warehouse Manager agent. You'll see:
- **LLM reasoning** — what the model decides to do
- **Tool calls** — which tools are invoked with what arguments
- **Tool outputs** — the results returned from each tool (including nested agent outputs)

In [ ]:
# ============================================================
# 📊 Detailed streaming with message-level granularity
# ============================================================
# stream_mode="messages" provides the most detailed view:
# Each message is yielded individually with metadata about
# which graph node produced it (model vs tools).
# ============================================================
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, ToolMessage

for msg, metadata in warehouse_manager.stream(inputs, stream_mode="messages"):
    # metadata["langgraph_node"] tells us which node produced this message
    node_name = metadata.get("langgraph_node")

    if node_name == "model":
        # Messages from the LLM reasoning step
        if msg.content:
            print(f"🧠 LLM: {msg.content}")
        # Check if the LLM decided to call a tool
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"🔧 LLM calls tool: {tool_call['name']} with args: {tool_call['args']}")

    elif node_name == "tools":
        # Messages from tool execution
        if isinstance(msg, ToolMessage) and msg.name:
            print(f"⚡ Called tool: {msg.name}")
            print(f"📋 Tool output: {msg.content}")

    else:
        # Catch any unexpected node types
        print(f"❓ Unexpected node name: {node_name}")

## 6. 🏋️ Exercise: Build Your Own Supervisor System

### 📝 Challenge

Using the patterns you've learned, build a simple supervisor-agent system:

1. **Create two agents:**
   - **Research Agent** — has a tool for searching information
   - **News Agent** — has a tool for getting latest news

2. **Create one supervisor agent** that delegates to the right agent based on user queries

3. **Connect both agents** and test the flow with sample queries

### 💡 Hints:
- Use the **agent-as-tool** pattern from the warehouse system (define tools that invoke sub-agents)
- Give each agent a clear system prompt about its specialization
- Test with queries like "What are the latest tech news?" vs. "Research the history of AI"

### 🎯 Bonus Challenge:
- Add memory/checkpointing so the supervisor remembers previous interactions
- Try using `langgraph-supervisor`'s `create_supervisor()` for a more elegant solution (see the Supervisor-Agent notebook!)

## 📝 Summary & Key Takeaways

### What We Built

A progressive learning journey through three levels of agent complexity:

| Part | What We Built | Key Concept |
|---|---|---|
| **Part 1** | Raw LLM call | Direct model invocation — no tools or state |
| **Part 2** | Simple weather agent | ReAct pattern — LLM + tools in a loop |
| **Part 3** | Hierarchical warehouse system | Agent-as-tool — nested delegation chain |

### 🔑 Core Concepts Reviewed

| Concept | What We Learned |
|---|---|
| **`create_agent()`** | Creates ReAct-style agents that can reason and use tools |
| **Agent-as-Tool Pattern** | A tool function can invoke another agent, creating hierarchies |
| **Bottom-Up Construction** | Build lower-level agents first since upper-level tools reference them |
| **`invoke()` vs `stream()`** | `invoke()` returns final result; `stream()` shows step-by-step execution |
| **`stream_mode="messages"`** | Most granular streaming — see every LLM decision and tool call |

### 🚀 Next Steps
- **Add more layers** — create a Floor Manager between Zone Manager and Warehouse Manager
- **Add multiple robots per zone** — the Zone Manager picks the right one based on capability
- **Add error handling** — what happens when a robot fails? How does the hierarchy recover?
- **Compare architectures** — see the **Supervisor-Agent** and **Swarm-Agent** notebooks for alternative approaches!

---

*💡 The agent-as-tool pattern is the foundation of hierarchical multi-agent systems. It's simple yet powerful — any agent can be wrapped as a tool for a higher-level agent!*